# SURE + Soustraction Spectrale

In [ ]:
from denoiser import pretrained #DEMUCS
from denoiser import demucs
import sys
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
from IPython.display import Audio, display

sys.path.append("../utils")
import noise
from tools import listen
from tools import denoise_spectral_sub
from noise import add_white_noise

In [ ]:
def sure_value(y, sigma, threshold, samplerate, denoise_func):
    _, fy = denoise_func(y, sigma, threshold, samplerate)
    L=min(len(y),len(fy))
    return np.mean((fy[:L]-y[:L])**2)

In [ ]:
#Import des fichiers sonores et verification de l'echantillonage
son1,fs1=sf.read('../../data/son1.wav')
print('Echantillonage 1 :',fs1)
son2,fs2=sf.read('../../data/son2.wav')
print('Echantillonage 2 :',fs2)
son3,fs3=sf.read('../../data/son3.wav')
print('Echantillonage 3 :',fs3)
son4,fs4=sf.read('../../data/son4.wav')
print('Echantillonage 4 :',fs4)

In [ ]:
#bruitage
tSNR=0
son1_b,s1=add_white_noise(son1,44100,tSNR,0,False)
listen(son1_b,44100)

In [ ]:
thresholds=np.linspace(0,5,100)

sure_min=np.inf
thresh_sure=0
for threshold in thresholds:
    sure=sure_value(son1_b,s1,threshold,44100,denoise_spectral_sub)
    if sure<sure_min:
        sure_min=sure
        thresh_sure=threshold
print(sure_min,thresh_sure)

### Utilisation de Deepinv

In [ ]:
import deepinv as dinv
import torch

In [ ]:
help(dinv.loss.SureGaussianLoss)

In [ ]:
import numpy as np
import torch
from torch import nn
import deepinv as dinv


class ThresholdDenoiserWrapper(nn.Module):
    """
    Wraps your threshold-based NumPy denoiser so DeepInv can call it as a model.

    Your denoiser must look like:
        denoise_fn(signal_np, threshold) -> denoised_np
    """

    def __init__(self, denoise_fn, threshold):
        super().__init__()
        self.denoise_fn = denoise_fn
        self.threshold = threshold

    def forward(self, y, *args, **kwargs):
        """
        y shape expected by DeepInv: [B, C, H, W]

        For mono audio, we use:
            [B, 1, 1, T]
        """
        device = y.device
        dtype = y.dtype

        y_cpu = y.detach().cpu().numpy()
        out = np.zeros_like(y_cpu)

        batch_size = y_cpu.shape[0]

        for i in range(batch_size):
            signal_1d = y_cpu[i, 0, 0, :]

            denoised_1d = self.denoise_fn(signal_1d, self.threshold)
            denoised_1d = np.asarray(denoised_1d, dtype=y_cpu.dtype)

            if denoised_1d.shape != signal_1d.shape:
                raise ValueError(
                    "Your denoise_fn must return the same shape as its input."
                )

            out[i, 0, 0, :] = denoised_1d

        return torch.from_numpy(out).to(device=device, dtype=dtype)


def deepinv_sure_threshold_search(
    y_np,
    denoise_fn,
    thresholds,
    sigma2,
    chunk_size=16384,
    device=None,
):
    """
    Uses DeepInv SureGaussianLoss to find the SURE-minimizing threshold.

    Parameters
    ----------
    y_np : np.ndarray
        Noisy mono audio signal, shape [T].

    denoise_fn : callable
        Your denoising function:
            denoise_fn(signal_np, threshold) -> denoised_np

    thresholds : array-like
        Thresholds to test.

    sigma2 : float
        Known white Gaussian noise variance.

    chunk_size : int
        Number of samples per chunk for SURE evaluation.

    Returns
    -------
    best_denoised : np.ndarray
        Denoised full signal using the SURE-best threshold.

    best_threshold : float
        Threshold minimizing DeepInv SURE.

    sure_values : np.ndarray
        SURE value for each threshold.
    """

    if device is None:
        device = dinv.utils.get_device()

    y_np = np.asarray(y_np, dtype=np.float32)

    if y_np.ndim != 1:
        raise ValueError("y_np must be a mono 1D NumPy array.")

    if sigma2 <= 0:
        raise ValueError("sigma2 must be positive.")

    sigma = float(np.sqrt(sigma2))

    # DeepInv denoising physics: y = x + Gaussian noise
    physics = dinv.physics.Denoising(
        noise_model=dinv.physics.GaussianNoise(sigma=sigma)
    )

    sure_loss = dinv.loss.SureGaussianLoss(sigma)

    # Pad signal so it can be split into equal chunks
    original_len = len(y_np)
    pad_len = (-original_len) % chunk_size

    if pad_len > 0:
        y_padded = np.pad(y_np, (0, pad_len), mode="reflect")
    else:
        y_padded = y_np

    # Shape for DeepInv: [B, C, H, W] = [chunks, 1, 1, chunk_size]
    chunks = y_padded.reshape(-1, chunk_size)
    y_tensor = torch.from_numpy(chunks[:, None, None, :]).to(device)

    sure_values = []

    for threshold in thresholds:
        model = ThresholdDenoiserWrapper(
            denoise_fn=denoise_fn,
            threshold=threshold,
        ).to(device)

        x_net = model(y_tensor)

        loss = sure_loss(
            x_net=x_net,
            y=y_tensor,
            physics=physics,
            model=model,
        )
        
        loss_scalar = loss.detach().mean().cpu().item()
        sure_values.append(loss_scalar)

    sure_values = np.asarray(sure_values)

    best_idx = int(np.argmin(sure_values))
    best_threshold = float(thresholds[best_idx])

    best_model = ThresholdDenoiserWrapper(
        denoise_fn=denoise_fn,
        threshold=best_threshold,
    ).to(device)

    with torch.no_grad():
        denoised_tensor = best_model(y_tensor)

    denoised_chunks = denoised_tensor.detach().cpu().numpy()[:, 0, 0, :]
    best_denoised = denoised_chunks.reshape(-1)[:original_len]

    return best_denoised, best_threshold, sure_values








In [ ]:
#bruitage
tSNR=0
son4_b,s4=add_white_noise(son4,44100,tSNR,0,False)
listen(son4_b,44100)

In [ ]:
class TorchSpectralSub(nn.Module):
    def __init__(
        self,
        sigma,
        threshold,
        fs,
        n_fft=1024,
        hop_length=256,
        eps=1e-12,
    ):
        super().__init__()

        self.sigma = float(sigma)
        self.threshold = float(threshold)
        self.fs = float(fs)

        self.n_fft = n_fft
        self.hop_length = hop_length
        self.eps = eps

    def forward(self, y, *args, **kwargs):
        """
        y shape: [B, 1, 1, T]
        output shape: [B, 1, 1, T]
        """

        B, C, H, T = y.shape
        y_audio = y[:, 0, 0, :]  # [B, T]

        window = torch.hann_window(
            self.n_fft,
            device=y.device,
            dtype=y.dtype,
        )

        Y = torch.stft(
            y_audio,
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            window=window,
            return_complex=True,
            center=True,
            normalized=False,
        )

        Sy = torch.abs(Y) ** 2

        # IMPORTANT FIX:
        # Expected STFT bin power of white noise:
        # E[|STFT(noise)|^2] = sigma^2 * sum(window^2)
        window_energy = torch.sum(window ** 2)
        Sb = (self.sigma ** 2) * window_energy

        # Same as your original:
        # Sx = max(Sy - threshold * Sb, 0)
        Sx = torch.clamp(Sy - self.threshold * Sb, min=0.0)

        Gain = Sx / (Sy + self.eps)

        X = Gain * Y

        x_hat = torch.istft(
            X,
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            window=window,
            length=T,
            center=True,
            normalized=False,
        )

        return x_hat[:, None, None, :]

In [ ]:
thresholds = np.linspace(0.1, 100, 100)
def fix_length(x, target_len):
    """
    Crop or zero-pad x so that len(x) == target_len.
    """
    x = np.asarray(x)

    if len(x) > target_len:
        return x[:target_len]

    if len(x) < target_len:
        return np.pad(x, (0, target_len - len(x)), mode="constant")

    return x
    
def my_denoiser_for_sure(signal, threshold):
    _,denoised = denoise_spectral_sub(
        y=signal,
        sigma=s4,
        threshold=threshold,
        fs=44100,
    )
    denoised = fix_length(denoised, len(signal))

    return denoised.astype(signal.dtype)

sigma2 = 0.03**2

denoised, best_threshold, sure_values = deepinv_sure_threshold_search(
    y_np=son4_b,
    denoise_fn=my_denoiser_for_sure,
    thresholds=thresholds,
    sigma2=s4**2,
    chunk_size=16384,
)

print("Best threshold:", best_threshold)

listen(denoised,44100)

In [ ]:
def deepinv_sure_spectral_sub_threshold_search(
    y_np,
    thresholds,
    sigma,
    fs,
    chunk_size=16384,
    n_fft=1024,
    hop_length=256,
    device=None,
):
    if device is None:
        device = dinv.utils.get_device()

    y_np = np.asarray(y_np, dtype=np.float32)

    if y_np.ndim != 1:
        raise ValueError("y_np must be a mono 1D NumPy array.")

    original_len = len(y_np)

    pad_len = (-original_len) % chunk_size

    if pad_len > 0:
        y_padded = np.pad(y_np, (0, pad_len), mode="reflect")
    else:
        y_padded = y_np

    chunks = y_padded.reshape(-1, chunk_size)

    # DeepInv expects image-like tensors
    y_tensor = torch.from_numpy(chunks[:, None, None, :]).to(device)

    physics = dinv.physics.Denoising(
        noise_model=dinv.physics.GaussianNoise(sigma=float(sigma))
    )

    sure_loss = dinv.loss.SureGaussianLoss(float(sigma),mc_iter=10)

    sure_values = []

    for threshold in thresholds:
        model = TorchSpectralSub(
            sigma=sigma,
            threshold=threshold,
            fs=fs,
            n_fft=n_fft,
            hop_length=hop_length,
        ).to(device)

        x_net = model(y_tensor)

        loss = sure_loss(
            x_net=x_net,
            y=y_tensor,
            physics=physics,
            model=model,
        )

        sure_scalar = loss.detach().mean().cpu().item()
        sure_values.append(sure_scalar)

    sure_values = np.asarray(sure_values)

    best_idx = int(np.argmin(sure_values))
    best_threshold = float(thresholds[best_idx])

    best_model = TorchSpectralSub(
        sigma=sigma,
        threshold=best_threshold,
        fs=fs,
        n_fft=n_fft,
        hop_length=hop_length,
    ).to(device)

    with torch.no_grad():
        x_best = best_model(y_tensor)

    denoised_chunks = x_best.detach().cpu().numpy()[:, 0, 0, :]
    denoised = denoised_chunks.reshape(-1)[:original_len]

    return denoised, best_threshold, sure_values

In [ ]:
tSNR = 0
son4_b, s4 = add_white_noise(son4, 44100, tSNR, 0, False)

thresholds = np.linspace(0.1, 20.0, 1000)

denoised2, best_threshold, sure_values = deepinv_sure_spectral_sub_threshold_search(
    y_np=son4_b,
    thresholds=thresholds,
    sigma=s4,
    fs=44100,
    chunk_size=16384,
    n_fft=1024,
    hop_length=256,
)

print("Best DeepInv SURE threshold:", best_threshold)

listen(denoised2, 44100)


In [ ]:
from metrics import SNR
print(SNR(son4,denoised2))

In [ ]:
tSNR = 5
son4_b, s4 = add_white_noise(son4, 44100, tSNR, 0, False)

thresholds = np.linspace(0.1, 20.0, 1000)

denoised2, best_threshold, sure_values = deepinv_sure_spectral_sub_threshold_search(
    y_np=son4_b,
    thresholds=thresholds,
    sigma=s1,
    fs=44100,
    chunk_size=16384,
    n_fft=1024,
    hop_length=256,
)

print("Best DeepInv SURE threshold:", best_threshold)

listen(denoised2, 44100)
print(SNR(son4,denoised2))

In [ ]:
tSNR = 10
son4_b, s4 = add_white_noise(son4, 44100, tSNR, 0, False)

thresholds = np.linspace(0.1, 40.0, 1000)

denoised2, best_threshold, sure_values = deepinv_sure_spectral_sub_threshold_search(
    y_np=son4_b,
    thresholds=thresholds,
    sigma=s4,
    fs=44100,
    chunk_size=16384,
    n_fft=1024,
    hop_length=256,
)

print("Best DeepInv SURE threshold:", best_threshold)

listen(denoised2, 44100)
print(SNR(son4,denoised2))

In [ ]:
tSNR = 15
son4_b, s4 = add_white_noise(son4, 44100, tSNR, 0, False)

thresholds = np.linspace(0.1, 40.0, 1000)

denoised2, best_threshold, sure_values = deepinv_sure_spectral_sub_threshold_search(
    y_np=son4_b,
    thresholds=thresholds,
    sigma=s4,
    fs=44100,
    chunk_size=16384,
    n_fft=1024,
    hop_length=256,
)

print("Best DeepInv SURE threshold:", best_threshold)

listen(denoised2, 44100)
print(SNR(son4,denoised2))